# Loading and Saving Data with PySpark

## Installing dependencies

In [6]:
%pip install pyspark pandas dotenv

import pyspark as py

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"


Note: you may need to restart the kernel to use updated packages.


## Create SparkSession & DataFrames

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("olist-bronze") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

In [8]:
df_orders = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_orders_dataset.csv")

df_products = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_products_dataset.csv")

df_customers = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_customers_dataset.csv")

df_order_items = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_order_items_dataset.csv")

df_order_payments = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_order_payments_dataset.csv")

df_geolocation = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_geolocation_dataset.csv")

df_reviews = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_order_reviews_dataset.csv")

df_sellers = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/olist_sellers_dataset.csv")

df_product_category_name_translation = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("mode", "PERMISSIVE") \
    .csv("../raw/product_category_name_translation.csv")

In [9]:
df_reviews.printSchema()       # types des colonnes
df_reviews.show(5)             # aperçu des données
df_reviews.count()             # nombre de lignes
df_reviews.describe().show()   # statistiques descriptives

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)

+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|           review_id|            order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------+--------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|7bc2406110b926393...|73fc7af87114b3971...|           4|                NULL|                  NULL| 2018-01-18 00:00:00|    2018-01-18 21:46:59|
|80e641a11e56f04c1...|a548910a1c6147796...|           5|         

+-------+--------------------+--------------------+------------------+--------------------+----------------------+
|summary|           review_id|            order_id|      review_score|review_comment_title|review_comment_message|
+-------+--------------------+--------------------+------------------+--------------------+----------------------+
|  count|               99224|               99224|             99224|               11568|                 40977|
|   mean|                NULL|                NULL|  4.08642062404257|3.165434995880696E10|     8.172413793103448|
| stddev|                NULL|                NULL|1.3475791311150984|5.625434750908947E11|    3.1175650470615843|
|    min|0001239bc1de2e33c...|00010242fe8c5a6d1...|                 1|                    |                    \n|
|    max|fffefe7a48d22f7b3...|fffe41c64501cc87c...|                 5|                 🔟 |  😡😡😡😡😡👎👎👎?...|
+-------+--------------------+--------------------+------------------+-------------------

In [10]:
from pyspark.sql.functions import col, sum as _sum

null_counts = df_orders.select([
    _sum(col(c).isNull().cast("int")).alias(c)
    for c in df_orders.columns
])
null_counts.show()

+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|order_id|customer_id|order_status|order_purchase_timestamp|order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+
|       0|          0|           0|                       0|              160|                        1783|                         2965|                            0|
+--------+-----------+------------+------------------------+-----------------+----------------------------+-----------------------------+-----------------------------+



## Identifier Clés de Jointures

In [12]:
for name, df in {
    "orders": df_orders,
    "products": df_products,
    "customers": df_customers,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "geolocation": df_geolocation,
    "order_reviews": df_reviews,
    "sellers": df_sellers,
    "category_translation": df_product_category_name_translation,
}.items():
    print(f"\n=== {name} ===")
    print(df.columns)


=== orders ===
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

=== products ===
['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']

=== customers ===
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

=== order_items ===
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

=== order_payments ===
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

=== geolocation ===
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

=== order_reviews ===
['review_id', 'order_id', 'review_score', 'review_comment_ti

In [14]:
from pyspark.sql.functions import countDistinct

# order_id présent dans orders, order_items, payments, reviews ?
print("order_id distincts dans orders :", df_orders.select(countDistinct("order_id")).collect()[0][0])
print("order_id distincts dans order_items :", df_order_items.select(countDistinct("order_id")).collect()[0][0])
print("order_id distincts dans payments :", df_order_payments.select(countDistinct("order_id")).collect()[0][0])
print("order_id distincts dans reviews :", df_reviews.select(countDistinct("order_id")).collect()[0][0])

# product_id présent dans order_items et products ?
print("\nproduct_id distincts dans order_items :", df_order_items.select(countDistinct("product_id")).collect()[0][0])
print("product_id distincts dans products :", df_products.select(countDistinct("product_id")).collect()[0][0])

# seller_id présent dans order_items et sellers ?
print("\nseller_id distincts dans order_items :", df_order_items.select(countDistinct("seller_id")).collect()[0][0])
print("seller_id distincts dans sellers :", df_sellers.select(countDistinct("seller_id")).collect()[0][0])

# customer_id présent dans orders et customers ?
print("\ncustomer_id distincts dans orders :", df_orders.select(countDistinct("customer_id")).collect()[0][0])
print("customer_id distincts dans customers :", df_customers.select(countDistinct("customer_id")).collect()[0][0])

order_id distincts dans orders : 99441
order_id distincts dans order_items : 98666
order_id distincts dans payments : 99440
order_id distincts dans reviews : 98673

product_id distincts dans order_items : 32951
product_id distincts dans products : 32951

seller_id distincts dans order_items : 3095
seller_id distincts dans sellers : 3095

customer_id distincts dans orders : 99441
customer_id distincts dans customers : 99441


In [15]:
# Y a-t-il des order_id dans order_items sans correspondance dans orders ?
orphan_items = df_order_items.join(df_orders, "order_id", "left_anti")
print("Articles sans commande correspondante :", orphan_items.count())

# Y a-t-il des product_id dans order_items sans correspondance dans products ?
orphan_products = df_order_items.join(df_products, "product_id", "left_anti")
print("Produits dans order_items absents de products :", orphan_products.count())

# Y a-t-il des seller_id dans order_items sans correspondance dans sellers ?
orphan_sellers = df_order_items.join(df_sellers, "seller_id", "left_anti")
print("Vendeurs dans order_items absents de sellers :", orphan_sellers.count())

Articles sans commande correspondante : 0
Produits dans order_items absents de products : 0
Vendeurs dans order_items absents de sellers : 0


## Save as Parquet

In [17]:
df_orders.write \
    .mode("overwrite") \
    .parquet("../data/bronze/orders/")

df_products.write \
    .mode("overwrite") \
    .parquet("../data/bronze/products/")

df_customers.write \
    .mode("overwrite") \
    .parquet("../data/bronze/customers/")

df_order_items.write \
    .mode("overwrite") \
    .parquet("../data/bronze/order_items/")

df_order_payments.write \
    .mode("overwrite") \
    .parquet("../data/bronze/order_payments/")

df_geolocation.write \
    .mode("overwrite") \
    .parquet("../data/bronze/geolocation/")

df_reviews.write \
    .mode("overwrite") \
    .parquet("../data/bronze/order_reviews/")

df_sellers.write \
    .mode("overwrite") \
    .parquet("../data/bronze/sellers/")
    
df_product_category_name_translation.write \
    .mode("overwrite") \
    .parquet("../data/bronze/product_category_name_translation/")